<div dir="rtl">

# الدرس 36: الانتشار العكسي باليد

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### المثال المحلول

</div>

In [ ]:
import numpy as np

x, y = 2.0, 1.0
w, b = np.array([0.5, -1.0]), np.array([0.0, 1.0])
v, c = np.array([1.0, 2.0]), 0.5

def forward(w, b, v, c):
    z = w * x + b
    h = np.maximum(0, z)
    y_hat = v @ h + c
    return z, h, y_hat, (y_hat - y) ** 2

z, h, y_hat, loss = forward(w, b, v, c)
print("forward : z =", z, " h =", h, " y_hat =", y_hat, " loss =", loss)

# backward, from the loss to the inputs
g_yhat = 2 * (y_hat - y)
g_v, g_c = g_yhat * h, g_yhat
g_h = g_yhat * v
g_z = g_h * (z > 0)          # ReLU: pass the gradient only where z > 0
g_w, g_b = g_z * x, g_z
print("grads   : w", g_w, " b", g_b, " v", g_v, " c", g_c)

lr = 0.1
w, b, v, c = w - lr * g_w, b - lr * g_b, v - lr * g_v, c - lr * g_c
print("new loss:", round(forward(w, b, v, c)[3], 4))

<div dir="rtl">

### فحص التدرّج

</div>

In [ ]:
def loss_of(params):
    w, b, v, c = params[:2], params[2:4], params[4:6], params[6]
    return forward(w, b, v, c)[3]

params = np.array([0.5, -1.0, 0.0, 1.0, 1.0, 2.0, 0.5])
backprop = np.array([2, 0, 1, 0, 1, 0, 1], dtype=float)   # from the hand example

eps = 1e-5
numeric = np.zeros_like(params)
for i in range(len(params)):
    p_plus, p_minus = params.copy(), params.copy()
    p_plus[i] += eps
    p_minus[i] -= eps
    numeric[i] = (loss_of(p_plus) - loss_of(p_minus)) / (2 * eps)

print("numeric :", numeric.round(6))
print("max difference:", np.abs(numeric - backprop).max())

<div dir="rtl">

### طبقات قابلة للتركيب

</div>

In [ ]:
class Linear:
    def __init__(self, d_in, d_out, rng):
        self.W = rng.normal(0, np.sqrt(2 / d_in), size=(d_in, d_out))
        self.b = np.zeros(d_out)

    def forward(self, X):
        self.X = X                       # keep the input for the backward pass
        return X @ self.W + self.b

    def backward(self, delta):
        self.dW = self.X.T @ delta
        self.db = delta.sum(axis=0)
        return delta @ self.W.T          # gradient for the previous layer

class ReLU:
    def forward(self, Z):
        self.mask = Z > 0
        return Z * self.mask

    def backward(self, delta):
        return delta * self.mask

In [ ]:
rng = np.random.default_rng(0)
layers = [Linear(3, 4, rng), ReLU(), Linear(4, 1, rng)]
X = rng.normal(size=(5, 3))
y = rng.normal(size=(5, 1))

def net_loss():
    out = X
    for layer in layers:
        out = layer.forward(out)
    return np.mean((out - y) ** 2), out

loss, out = net_loss()
delta = 2 * (out - y) / len(X)           # derivative of the mean squared error
for layer in reversed(layers):
    delta = layer.backward(delta)

W = layers[0].W
numeric = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        W[i, j] += 1e-5; lp = net_loss()[0]
        W[i, j] -= 2e-5; lm = net_loss()[0]
        W[i, j] += 1e-5
        numeric[i, j] = (lp - lm) / 2e-5

rel_error = np.abs(numeric - layers[0].dW).max() / np.abs(numeric).max()
print("dW shape:", layers[0].dW.shape, " relative error:", f"{rel_error:.1e}")

<div dir="rtl">

## تمرين: مثال جديد بنفس الشبكة

نفس الشبكة، ونفس القيم الابتدائية. لكن المثال الآن: المدخل $x = -1$، والهدف $y = 2$. ومعدل التعلّم 0.05.

1. احسب التمرير الأمامي والخسارة.
2. احسب كل التدرّجات السبعة.
3. طبّق خطوة التعلّم، واحسب الخسارة الجديدة.
4. أي عصبون "مات" لهذا المثال هذه المرة؟

</div>

In [ ]:
# اكتب حلّك هنا
